In [ ]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

# Aave V3.1 — Dune table fetcher

Cell 1 loads `DUNE_API_KEY` (from `.env` locally, or from an injected environment
variable when running in the extraction container — `load_api_key` handles both).

Cell 2 fetches each query id in `QUERY_IDS`. Ids mirror **`context/api.md`**, the
registry — keep the two in sync when a query is re-run on Dune and gets a new id.
All fetching logic lives in `dune_fetch.py` (imports: `requests`, `pandas`,
`python-dotenv`). Fetches read Dune's *stored* results, so they spend no execution
credits, and each save is versioned as
`query_result_data/query_result_data_{id}_{utc_stamp}.csv`.

In [ ]:
# Cell 1 — broadcast the .env API key into this kernel's environment
from dune_fetch import load_api_key

load_api_key()
print("DUNE_API_KEY loaded \u2713")

In [ ]:
# Cell 2 — fetch each table and save it as <table_name>_<query_id>_<stamp>.csv
from dune_fetch import fetch_query_table
from IPython.display import display

# Keys are the TABLE NAMES used on disk and in data_validation.TABLE_LABELS;
# ids mirror context/api.md.
#
# 2026-07-25 fetch status — TWO separate blockers, do not conflate them:
#
#   402 Payment Required : supply_withdraw, borrow_repay, reserve_config
#       Ids are CORRECT — all three fetched successfully earlier the same day.
#       The Dune API quota is exhausted (supply_withdraw died at offset=32000,
#       i.e. mid-pagination). fetch_query_table spends no EXECUTION credits, but
#       Dune still meters API requests / data transfer, and re-pulling the 25 MB
#       reserve_config several times burned through it. These just need a retry
#       after the quota resets — do NOT recreate them on Dune.
#
#   404 Not Found : reserve_state_rates, liquidation, flashloan, user_account,
#                   collateral_toggle, oracle_price, all decimal_reference_*
#       Not visible to the CURRENT key — they belong to the other Dune account.
#       Swap DUNE_API_KEY to that account's key to fetch them.
#
# NOTE the oracle name says "6h" while the query is now 2h-grain. Keep it: both
# TABLE_LABELS and normalize.ipynb's glob look for
# oracle_price_usd_eth_weth_6h_*.csv. Renaming means changing those too.
QUERY_IDS = {
    "supply_withdraw":                           7702138,
    "borrow_repay":                              7798273,
    "reserve_state_rates":                       7711042,
    "liquidation":                               7798339,
    "flashloan":                                 7798349,
    "user_account":                              7798351,
    "collateral_toggle":                         7798372,
    "oracle_price_usd_eth_weth_6h":              7798226,
}

# Skipped for now. The decimal references are static per-asset decimals — values
# don't change, the 2026-06-23 CSVs are already in query_result_data/, and those
# are what normalize.ipynb reads. reserve_config is here only because of the 402
# quota (its id is valid); re-enable it once the quota resets.
DISABLED_QUERY_IDS = {
    "decimal_reference_part1":                   7711171,   # 404
    "decimal_reference_part2":                   7711265,
    "decimal_reference_part3":                   7711276,
    "decimal_reference_part4":                   7711290,
    "decimal_reference_part5":                   7711298,
    "decimal_reference_part6":                   7711304,
    "decimal_reference_part9_collateral_toggle": 7711307,
    "reserve_config":                            7804264,   # 402 quota, id is fine
}

# Per-query error handling: one dead id must not abort the rest. Failures are
# collected and reported together, so a single run shows the whole picture and
# distinguishes 402 (quota) from 404 (wrong key).
tables, failures = {}, {}
for table_name, query_id in QUERY_IDS.items():
    try:
        df, csv_path = fetch_query_table(query_id, table_name=table_name)
    except Exception as exc:
        failures[table_name] = f"{query_id}: {type(exc).__name__}: {exc}"
        print(f"FAILED  {table_name} ({query_id}) -> {type(exc).__name__}")
        continue
    print(f"ok      {table_name} ({query_id}): {len(df)} rows -> {csv_path}")
    display(df)
    tables[table_name] = df

print(f"\nfetched {len(tables)}/{len(QUERY_IDS)} tables"
      f"  ({len(DISABLED_QUERY_IDS)} skipped as disabled)")
if failures:
    print("failures:")
    for name, why in failures.items():
        print(f"  {name}: {why}")